# Временные признаки

В этом ноутбуке из столбца `date` создаются признаки для прогнозирования энергопотребления: час, день недели, признак выходного дня и циклическое представление часа.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Загрузка данных

Ноутбук поддерживает запуск как из папки `notebooks/`, так и из корня репозитория.

In [2]:
data_paths = [
    Path('../data/energydata_complete.csv'),
    Path('data/energydata_complete.csv'),
]
data_path = next((path for path in data_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError('Не найден data/energydata_complete.csv')

df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'], errors='raise')
df[['date', 'Appliances']].head()

,date,Appliances
0,2016-01-11 17:00:00,60
1,2016-01-11 17:10:00,60
2,2016-01-11 17:20:00,50
3,2016-01-11 17:30:00,50
4,2016-01-11 17:40:00,60


## Создание признаков

`weekday` принимает значения от 0 (понедельник) до 6 (воскресенье). `is_weekend` равен 1 для субботы и воскресенья. Пара `sin_hour` и `cos_hour` помогает модели понимать, что 23:00 и 00:00 находятся рядом в суточном цикле.

In [3]:
df['hour'] = df['date'].dt.hour
df['weekday'] = df['date'].dt.weekday
df['is_weekend'] = (df['weekday'] >= 5).astype('int8')

df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24)
df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24)

temporal_features = ['hour', 'weekday', 'is_weekend', 'sin_hour', 'cos_hour']
df[['date', *temporal_features]].head(10)

,date,hour,weekday,is_weekend,sin_hour,cos_hour
0,2016-01-11 17:00:00,17,0,0,-0.965926,-2.588190e-01
1,2016-01-11 17:10:00,17,0,0,-0.965926,-2.588190e-01
2,2016-01-11 17:20:00,17,0,0,-0.965926,-2.588190e-01
3,2016-01-11 17:30:00,17,0,0,-0.965926,-2.588190e-01
4,2016-01-11 17:40:00,17,0,0,-0.965926,-2.588190e-01
5,2016-01-11 17:50:00,17,0,0,-0.965926,-2.588190e-01
6,2016-01-11 18:00:00,18,0,0,-1.000000,-1.836970e-16
7,2016-01-11 18:10:00,18,0,0,-1.000000,-1.836970e-16
8,2016-01-11 18:20:00,18,0,0,-1.000000,-1.836970e-16
9,2016-01-11 18:30:00,18,0,0,-1.000000,-1.836970e-16


## Проверка результата

Проверяем, что новые признаки не содержат пропусков, имеют ожидаемые границы значений и могут быть переданы на этап моделирования.

In [4]:
assert df[temporal_features].isna().sum().sum() == 0, 'В новых признаках есть пропуски'
assert df['hour'].between(0, 23).all(), 'Некорректное значение часа'
assert df['weekday'].between(0, 6).all(), 'Некорректное значение дня недели'
assert set(df['is_weekend'].unique()).issubset({0, 1}), 'Некорректный признак выходного дня'

df[temporal_features].describe().T

,count,mean,std,min,25%,50%,75%,max
hour,19735.0,11.502002,6.921953,0.0,6.000000,1.200000e+01,17.000000,23.0
weekday,19735.0,2.977249,1.985617,0.0,1.000000,3.000000e+00,5.000000,6.0
is_weekend,19735.0,0.277274,0.447664,0.0,0.000000,0.000000e+00,1.000000,1.0
sin_hour,19735.0,-0.000344,0.707236,-1.0,-0.707107,0.000000e+00,0.707107,1.0
cos_hour,19735.0,-0.000079,0.707014,-1.0,-0.707107,-1.836970e-16,0.707107,1.0


### Итог

В таблицу добавлены пять временных признаков. На следующих этапах их можно использовать вместе с показаниями датчиков и внешними метеоданными для обучения модели.